# Can the mtanh baseline fit be trusted where the scan acts?

The sparse-grid axes are scale factors on mtanh fit parameters, so a bad fit
does not produce a bad point — it silently redefines what every axis value
means, for the whole scan. This notebook measures the fit where that matters
and nowhere else, then converts the literature bounds into scale factors for
each discharge.

**One parameterization: `fit_mtanh_full`** (Stefanikova 2016, axis-to-SOL in
one formula), driven by `apply_mtanh_full`. The pedestal-only Bruncrona form
was carried alongside it through several passes and lost on every discharge
that separated them — 129038 `ne` 13.3% vs 0.6%, and no case where the
pedestal form won by more than noise. One knob set across every axis and
discharge is worth more to a sparse-grid campaign than a per-axis best form,
so the comparison is retired and `ped` no longer appears here.

**The verdict window is per discharge, and measured.** A single global
`rho_tor` window is the wrong test twice over: `rms_relative` over the whole
profile scores a fit on core structure the scan never touches, and a fixed
0.60–0.95 band asks 132588 (whose $p_e$ is already falling at
$\rho_t\approx0.56$) and 129015 (0.86) the same question. So each discharge
gets its own pedestal region, taken as the quarter-maximum band of
$|\nabla p_e|$ from the data — not from the fit, which is the thing under
test — widened to cover the radii GENE is actually run at. The rms inside that window is what licenses the scale
factors.

Known consequence of standardizing on the full form, and why it is acceptable:
where the fitted SOL floor `b_sol` comes out at zero (Te really does go to ~0
in the SOL), `scale_height` multiplies the whole pedestal component uniformly
and $a/L$ is exactly invariant under multiplication — the height knob moves
the profile but cannot move the drive. Te drive then comes from the WIDTH
axis, which is what Boyle's $\Delta T_e$ bounds are for. Flagged per discharge
in the coverage section.

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import medfilt

from TPED.projects.discharge_tools.src.discharge_data import DischargeData
from TPED.projects.discharge_tools.src.discharge_physics import DischargePhysics
from TPED.projects.discharge_tools.src.transforms.mtanh_transforms import fit_mtanh_full

# Both machines, first one that exists wins — same notebook runs on NERSC and
# on the laptop. Add a path rather than replacing one.
DISCHARGE_ROOT = next(
    (d for d in [r"/global/homes/j/joeschm/data/ST_research/NSTXU_discharges",
                 r"C:/Users/joesc/git/ST_research/NSTXU_discharges"]
     if os.path.isdir(d)), None)
if DISCHARGE_ROOT is None:
    raise FileNotFoundError("no discharge root found; add this machine's path")

# 129038's directory holds five pfiles, so auto-discovery refuses to guess.
DISCHARGES = {129015: {}, 129038: {"pfile": "p129038.00400"},
              132543: {}, 132588: {}}

# Profiles the axes act on, plus the ones that follow through quasineutrality —
# a fit fine for ne and wrong for ni still reaches CHEASE.
VARS = ["Te", "Ti", "ne", "ni"]

FIT_KWARGS = dict(pedestal_weight=8.0)   # matches the campaign
TRUST = 0.01                             # rms <= 1% of profile range, in-window

# Where GENE is actually run. 132543/132588's radii are the q=4 and q=5
# surfaces and sit inside the pedestal top, so the verdict window has to reach
# them even when the pedestal proper starts further out.
ANALYSIS_RADII = {129015: (0.85,), 129038: (0.85,),
                  132543: (0.736, 0.825), 132588: (0.736, 0.825)}

SCALE_BOX = (0.7, 1.3)     # campaign scale box, for the coverage section

# Pedestal-region detection, on |grad pe| from the DATA (the fit is under test).
# A quarter-maximum band rather than half: half-max cut 132588 off at 0.625 when
# its pe profile is visibly falling from ~0.55, because a single-point spike at
# 0.740 sets the maximum the threshold is taken against. The median filter kills
# that spike; the looser fraction makes the window conservative, which is the
# right direction for a test the fit has to pass.
GRAD_FRAC   = 0.25         # quarter-maximum band around the peak gradient
GRAD_MEDFILT = 5           # points; spike rejection before thresholding
GRAD_SEARCH = (0.50, 0.999)   # 132543's core gradient dominates below this
WINDOW_CAP  = 0.99         # never score past here: pfile SOL is untrustworthy
WINDOW_MIN_HI = 0.95       # always score at least out to here

# fit_mtanh_full bounds b_pos to [0.85, 0.999]. A fit landing exactly on 0.85
# has hit the bound, not found the pedestal — reported, not silently used.
BPOS_BOUND = 0.85

## Fit, and locate each discharge's pedestal

One pass: fit `Te`, `Ti`, `ne`, `ni` and the derived `pe = ne*Te` with
`fit_mtanh_full`, then set the windows.

- **pedestal region** — quarter-maximum band of $|\nabla p_e|$, computed with
  TPED's `gradient_length` (4th-order non-uniform) on the measured profile and
  median-filtered before thresholding. Data-driven on purpose: deriving it from
  `b_pos ± 2 b_width` would let a bad fit choose the window that judges it, and
  three of these four discharges return `b_pos` pinned on the fitter's 0.85
  bound. Quarter rather than half-maximum because a half-max cut on the raw
  gradient put 132588's inner edge at 0.625 while its $p_e$ is visibly falling
  from ~0.55 — one spiky point at 0.740 was setting the threshold the rest of
  the band was measured against. The looser band errs wide, which is the safe
  direction for a test the fit has to pass.
- **verdict window** — the pedestal region widened to cover `ANALYSIS_RADII`
  and out to at least 0.95, capped at 0.99.

`Ti` and `ni` are fitted, plotted and scored alongside the axis variables: no
scan axis names them, but quasineutrality and the closure carry them into
CHEASE regardless.

Errors are normalized by profile RANGE, not by $y$: a relative error against a
value going to ~0 at the separatrix explodes there and swamps the region under
test.

In [ ]:
def load(shot):
    """DischargePhysics for one shot, with pe = ne*Te added as a variable so
    TPED's fitter and gradient_length see it like any other profile."""
    d = os.path.join(DISCHARGE_ROOT, str(shot))
    kw = {"input_dir": d}
    if DISCHARGES[shot].get("pfile"):
        kw["pfile"] = os.path.join(d, DISCHARGES[shot]["pfile"])
    phys = DischargePhysics(DischargeData(**kw))
    ds = phys.ds.copy()
    ds["pe"] = phys.ds["ne"] * phys.ds["Te"]
    return DischargePhysics(ds)


def ped_region(phys, var="pe", frac=GRAD_FRAC, search=GRAD_SEARCH):
    """Quarter-maximum band of |grad var| — the pedestal, from the data.

    Returns (lo, peak, hi) in rho_tor. Median-filtered first so one noisy point
    cannot set the maximum the threshold is measured against, and contiguous
    around the peak, so a second steep feature further in does not silently
    annex the window.
    """
    x = np.asarray(phys.rhot.values)
    y = np.asarray(phys.ds[var].values, dtype=float)
    g = medfilt(np.abs(y / np.asarray(phys.gradient_length(var).values)),
                GRAD_MEDFILT)
    m = (x >= search[0]) & (x <= search[1])
    xs, gs = x[m], g[m]
    i = int(np.nanargmax(gs))
    lo = i
    while lo > 0 and gs[lo - 1] >= frac * gs[i]:
        lo -= 1
    hi = i
    while hi < len(gs) - 1 and gs[hi + 1] >= frac * gs[i]:
        hi += 1
    return float(xs[lo]), float(xs[i]), float(xs[hi])


def fit_record(phys, var, rec, **kw):
    """Fit one profile and return the fields the rest of the notebook reads."""
    x = np.asarray(phys.rhot.values)
    try:
        profile, meta = fit_mtanh_full(phys.ds, var, **kw)
    except Exception as exc:
        return {"error": f"{type(exc).__name__}: {exc}"}
    yhat = np.asarray(profile(x), dtype=float)
    fp = meta["fit_params"]
    return {"yhat": yhat, "err": (yhat - rec["y"]) / rec["scale"],
            "rms_global": meta["rms_relative"], "params": fp,
            "pinned": abs(fp["b_pos"] - BPOS_BOUND) < 1e-6}


def window_stats(x, err, win):
    """rms and max |error| inside a window, as a fraction of profile range."""
    e = err[(x >= win[0]) & (x <= win[1])]
    return float(np.sqrt(np.mean(e ** 2))), float(np.max(np.abs(e)))


results = {}
for shot in DISCHARGES:
    phys = load(shot)
    x = np.asarray(phys.rhot.values)
    lo, pk, hi = ped_region(phys)
    win = (min([lo] + list(ANALYSIS_RADII[shot])),
           min(max(hi, WINDOW_MIN_HI), WINDOW_CAP))
    entry = {"phys": phys, "x": x, "ped": (lo, pk, hi), "win": win, "vars": {}}
    for var in VARS + ["pe"]:
        y = np.asarray(phys.ds[var].values, dtype=float)
        rec = {"y": y, "scale": float(np.max(y) - np.min(y)) or 1.0,
               "kwargs": dict(FIT_KWARGS)}
        rec.update(fit_record(phys, var, rec, **FIT_KWARGS))
        entry["vars"][var] = rec
        if "err" not in rec:
            print(f"  {shot} {var}: FIT FAILED — {rec['error']}")
    results[shot] = entry

shots = sorted(results)
varlist = [v for v in VARS + ["pe"] if any(v in e["vars"] for e in results.values())]

print(f"{'shot':>7}  {'pedestal region':>17} {'peak':>6}  {'verdict window':>15}"
      f"  {'GENE radii':>13}")
print("-" * 70)
for s in shots:
    lo, pk, hi = results[s]["ped"]
    w = results[s]["win"]
    radii = ",".join(f"{r:.3f}" for r in ANALYSIS_RADII[s])
    print(f"{s:>7}  {lo:>8.3f}-{hi:<8.3f} {pk:>6.3f}  {w[0]:>7.3f}-{w[1]:<7.3f}"
          f"  {radii:>13}")
print("\npedestal region = quarter-max band of |grad pe| from the data; "
      "verdict window = that band widened to the GENE radii and to 0.95, "
      f"capped at {WINDOW_CAP}")

## The verdict — error inside each discharge's own window

`rms_win` / `max_win` are taken over the verdict window; `rms_ped` over the
pedestal region alone; `rms_in` over the stretch immediately inboard of the
window (0.50 to its inner edge); `rms_global` is the whole-profile number the
first pass used, kept for contrast. Where `rms_global` and `rms_win` diverge,
the global number was measuring something the scan does not touch.

`rms_in` is not part of the verdict but is the honest check on it: a profile
that passes in-window while carrying several percent just inboard — 129015
`Ti` is the case here — is being flattered by a narrow window, and for
`Ti`/`ni` that error still reaches CHEASE through the closure.

`b_pos` is printed because a value sitting exactly on 0.85 is the fitter's
lower bound, not a located pedestal — that fit can still score well and still
be the wrong shape to scale, and it is the failure mode behind the runaway
`ne_width_scale` numbers further down.

In [ ]:
rows = []
for shot in shots:
    e = results[shot]
    x, win, ped = e["x"], e["win"], e["ped"][0::2]
    for var in varlist:
        f = e["vars"].get(var, {})
        if "err" not in f:
            rows.append({"shot": shot, "var": var, "rms_win": np.nan,
                         "max_win": np.nan, "rms_ped": np.nan, "rms_in": np.nan,
                         "rms_global": np.nan, "b_pos": np.nan,
                         "pinned": False, "ok": False,
                         "note": f.get("error", "")[:40]})
            continue
        rms, mx = window_stats(x, f["err"], win)
        # Error immediately INBOARD of the window, GRAD_SEARCH[0] to the window
        # edge. Not part of the verdict -- no axis acts there -- but a large
        # number here means the fit is only good because the window is narrow,
        # and for Ti/ni it still reaches CHEASE through the closure.
        inb = (window_stats(x, f["err"], (GRAD_SEARCH[0], win[0]))[0]
               if win[0] > GRAD_SEARCH[0] else np.nan)
        rows.append({"shot": shot, "var": var, "rms_win": rms, "max_win": mx,
                     "rms_ped": window_stats(x, f["err"], ped)[0], "rms_in": inb,
                     "rms_global": f["rms_global"], "b_pos": f["params"]["b_pos"],
                     "pinned": f["pinned"], "ok": rms <= TRUST, "note": ""})

hdr = (f"{'shot':>7} {'var':<4} {'rms_win':>8} {'max_win':>8} {'rms_ped':>8} "
       f"{'rms_in':>7} {'rms_global':>10} {'b_pos':>7} {'verdict':>8}")
print(hdr); print("-" * len(hdr))
for r in rows:
    nan = lambda v, w: (f"{v*100:>{w-1}.2f}%" if v == v else f"{'--':>{w}}")
    print(f"{r['shot']:>7} {r['var']:<4} {nan(r['rms_win'],8)} {nan(r['max_win'],8)} "
          f"{nan(r['rms_ped'],8)} {nan(r['rms_in'],7)} {nan(r['rms_global'],10)} "
          f"{r['b_pos']:>7.4f} {'TRUST' if r['ok'] else 'reject':>8}"
          + ("   b_pos ON BOUND" if r["pinned"] else "")
          + (f"   {r['note']}" if r["note"] else ""))

print(f"\nTRUST: rms inside the discharge's verdict window <= {TRUST:.0%} of range")
print(f"{'shot':>7}  {'Te':>7} {'ne':>7} {'pe':>7}   axes usable?")
print("-" * 48)
verdict = {}
for shot in shots:
    v = {var: next((r["rms_win"] for r in rows
                    if r["shot"] == shot and r["var"] == var), np.nan)
         for var in ("Te", "ne", "pe")}
    ok = all(t == t and t <= TRUST for t in v.values())
    verdict[shot] = {**{f"{k}_rms_win": t for k, t in v.items()},
                     "window": list(results[shot]["win"]),
                     "ped_region": list(results[shot]["ped"][0::2]), "ok": ok}
    print(f"{shot:>7}  " + " ".join(f"{t*100:>6.2f}%" for t in v.values())
          + f"   {'yes' if ok else 'NO — refit before scaling'}")

with open("mtanh_fit_quality.json", "w") as fh:
    json.dump({"form": "full", "trust_threshold": TRUST,
               "rows": rows, "verdict": verdict}, fh, indent=1, default=str)
print("\nwritten: mtanh_fit_quality.json")

## Look at it

Three figures, in the order they should be read.

1. **Pedestal region** — data vs fit over each discharge's own window (shaded =
   verdict window, dotted = GENE radii, dash-dot = peak $|
abla p_e|$). What
   disqualifies a fit here is error *rising inside the shaded band*.
2. **Full radius** — the same fits from axis to SOL. This is the profile
   `apply_mtanh_full` writes and **CHEASE-BS reshapes on all of it**, so a
   reconstruction that nails the pedestal and wanders on axis is a defect the
   window score is structurally unable to fail. Watch for a core that misses the
   measured on-axis value, a non-monotonic bump, or the fit crossing the data
   inboard of the pedestal top. Each panel prints both numbers, `win` and `all`,
   so the two views cannot be read apart.
3. **The verdict** — one bar per profile, threshold drawn.

The bottom row of each grid is the residual against radius, always full radius:
a fit poor in the core and sound in the window is visibly that rather than
assumed to be. Core error alone is not disqualifying for the *scale factors* —
no axis acts there — but it is what CHEASE-BS sees.

In [ ]:
COL = {"Te": "tab:red", "Ti": "tab:orange", "ne": "tab:blue",
       "ni": "tab:cyan", "pe": "tab:purple"}
PLOT_VARS = ["Te", "Ti", "ne", "ni", "pe"]   # Ti/ni included: they reach
#   CHEASE through quasineutrality and the closure even though no axis names them

def profile_grid(xlim, title):
    """Data vs fit, one row per profile, one column per discharge.

    xlim=None zooms on each discharge's own window; xlim=(0, 1) draws the whole
    radius, which is what apply_mtanh_full hands to CHEASE-BS.
    """
    fig, axes = plt.subplots(len(PLOT_VARS) + 1, len(shots),
                             figsize=(3.5 * len(shots), 2.2 * (len(PLOT_VARS) + 1)),
                             squeeze=False)
    for j, shot in enumerate(shots):
        e = results[shot]
        x, win = e["x"], e["win"]
        lo, pk, hi = e["ped"]
        xl = xlim or (win[0] - 0.15, 1.0)
        for i, var in enumerate(PLOT_VARS):
            ax = axes[i][j]
            f = e["vars"].get(var, {})
            m = (x >= xl[0]) & (x <= xl[1])
            ax.plot(x[m], f["y"][m], ".", ms=3, color="0.55", label="data")
            if "yhat" in f:
                ax.plot(x[m], f["yhat"][m], "-", lw=1.5, color=COL[var],
                        label="full fit")
                r = window_stats(x, f["err"], win)[0]
                ax.text(0.03, 0.08,
                        f"win {r*100:.2f}%  |  all {f['rms_global']*100:.2f}%",
                        fontsize=7, transform=ax.transAxes,
                        color="green" if r <= TRUST else "tab:red")
            ax.axvspan(*win, color="tab:green", alpha=0.10)
            ax.axvline(pk, color="k", lw=0.7, ls="-.", alpha=0.5)
            for r0 in ANALYSIS_RADII[shot]:
                ax.axvline(r0, color="k", lw=0.8, ls=":", alpha=0.8)
            ax.set_xlim(*xl)
            ax.tick_params(labelsize=7)
            if i == 0:
                ax.set_title(f"{shot}   ped {lo:.2f}-{hi:.2f}", fontsize=10)
            if j == 0:
                ax.set_ylabel(var)
        # residual row, always full radius
        ax = axes[-1][j]
        for var in varlist:
            f = e["vars"].get(var, {})
            if "err" in f:
                ax.plot(x, 100 * f["err"], lw=1.1, color=COL[var], label=var,
                        alpha=0.9)
        ax.axvspan(*win, color="tab:green", alpha=0.10)
        ax.axhline(0, color="k", lw=0.6)
        ax.set_xlim(0, 1.0); ax.set_ylim(-8, 8)
        ax.set_xlabel("rho_tor"); ax.tick_params(labelsize=7)
        if j == 0:
            ax.set_ylabel("(fit - data)/range [%]")
    axes[0][-1].legend(fontsize=6)
    axes[-1][-1].legend(fontsize=6, ncol=2)
    fig.suptitle(title)
    plt.tight_layout()
    return fig


profile_grid(None, "Pedestal region — shaded: verdict window, dotted: GENE "
                   "radii, dash-dot: peak |grad pe|")
plt.show()

# The whole radius: this is the profile apply_mtanh_full actually writes, and
# CHEASE-BS reshapes on all of it. A reconstruction that tracks the pedestal and
# wanders on axis is a fit this notebook's window score cannot fail -- check it
# here instead. Watch for: a core that misses the measured on-axis value, a
# non-monotonic bump, or a crossing between the fit and the data inboard of the
# pedestal top.
profile_grid((0.0, 1.0), "Full radius — what apply_mtanh_full hands to CHEASE-BS")
plt.show()

fig, ax = plt.subplots(figsize=(10, 3.4))
labels = [f"{r['shot']} {r['var']}" for r in rows]
ax.bar(np.arange(len(rows)), [100 * r["rms_win"] for r in rows], 0.6,
       color=[COL[r["var"]] for r in rows])
ax.axhline(100 * TRUST, color="k", ls="--", lw=1.2, label=f"trust {TRUST:.0%}")
ax.set_xticks(np.arange(len(rows)))
ax.set_xticklabels(labels, fontsize=6, rotation=90)
ax.set_yscale("log"); ax.set_ylabel("rms in verdict window [% of range]")
ax.legend(fontsize=8)
ax.set_title("per-discharge verdict — below the dashed line is usable")
plt.tight_layout(); plt.show()

### If a profile is rejected

In order of effort; re-run the cells above after each, the verdict table is the
arbiter.

1. **Raise `pedestal_weight`** (currently 8) — upweights points inside
   `ped_threshold`–`edge_threshold`, buying pedestal accuracy at the core's
   expense, which is the trade this window says we want.
2. **Lower `ped_threshold`** (default 0.85) — for the discharges whose `b_pos`
   is pinned on the 0.85 bound, the pedestal genuinely sits further in and the
   fitter is not allowed to look there. This is the first thing to try for
   129038 and 132588.
3. **Supply explicit `p0`** — eight parameters `[b_height, b_sol, b_pos,
   b_width, b_slope, a_height, a_width, a_exp]`. Seed `b_pos` and `b_width`
   from the measured peak-gradient location printed above; those two are what
   the auto-guess gets wrong.

**Tried and rejected (2026-08-22): per-profile tuning of `ped_threshold`,
seeded `p0`, and `b_pos` bounds freed to the measured region, swept over
`pedestal_weight` 8–64.** It worked on the metric this notebook reports — every
profile passed in-window, worst rms 0.74%, 129038 `pe` went 4.54% to 0.21% —
and it was reverted anyway. Freeing `b_pos` inward (0.68–0.76 on 129038 and
132588) buys pedestal accuracy by letting the mtanh reinterpret half the
profile as pedestal, and the resulting reconstruction is distorted outside the
pedestal. **CHEASE-BS consumes the whole reconstructed profile, not the window
this notebook scores**, so a fit that wins in-window and wanders in the core is
worse for the campaign than one that is mediocre in-window and stable
everywhere. The single campaign setting — `pedestal_weight=8`, default
`ped_threshold`, default bounds — is what is used, and rejections are addressed
by steps 1–3 one profile at a time rather than by an automated sweep.

## What span do the axes actually buy? — $a/L_{p_e}$ at the analysis radius

Everything below is the scaling half, unchanged in intent and now single-form.
The only quantity that reaches the physics is what GENE computes from the
profiles it is handed, and for drive that is the normalized inverse scale
length — measured with TPED's `gradient_length`, not a local difference.

A flat line is an axis that cannot move the drive at that radius: the scan is
a no-op there whatever the fit quality. Non-monotonicity is a separate problem
— the sparse grid assumes the QoI is smooth in the axes.

In [ ]:
def a_over_L_pe(phys, x0):
    """a/L for pe at one radius, from TPED's 4th-order non-uniform derivative."""
    ds = phys.ds.copy()
    ds["pe"] = phys.ds["ne"] * phys.ds["Te"]
    p = DischargePhysics(ds)
    x = np.asarray(p.rhot.values)
    L = np.asarray(p.gradient_length("pe").values)
    return float(1.0 / L[int(np.argmin(np.abs(x - x0)))])


scales = np.linspace(SCALE_BOX[0], SCALE_BOX[1], 7)
curves, span_rows = {}, []
for shot in shots:
    phys = results[shot]["phys"]
    fits = {v: fit_mtanh_full(phys.ds, v, **FIT_KWARGS)[0] for v in ("Te", "ne")}
    for x0 in ANALYSIS_RADII[shot]:
        for axis in ("Te", "ne"):
            vals = [a_over_L_pe(
                phys.apply_mtanh_full(axis, fit=fits[axis], scale_height=s,
                                      enforce_quasineutrality=True, qz=6.0), x0)
                for s in scales]
            curves[(shot, x0, axis)] = vals
            base = vals[len(scales) // 2]
            span_rows.append({"shot": shot, "x0": x0, "axis": axis,
                              "span": max(vals) - min(vals),
                              "span_pct": 100 * (max(vals) - min(vals)) / abs(base)})

print(f"a/L_pe span across scale_height {SCALE_BOX[0]}-{SCALE_BOX[1]}")
print(f"{'shot':>7} {'x0':>6} {'axis':<4} {'span':>8} {'span %':>8}  {'b_sol':>11}")
print("-" * 52)
for r in span_rows:
    bs = results[r["shot"]]["vars"][r["axis"]]["params"]["b_sol"]
    flag = "  <-- b_sol=0: height axis cannot move a/L" if abs(bs) < 1e-9 else ""
    print(f"{r['shot']:>7} {r['x0']:>6.3f} {r['axis']:<4} {r['span']:>8.3f} "
          f"{r['span_pct']:>7.1f}% {bs:>11.4g}{flag}")

pairs = [(s, x0) for s in shots for x0 in ANALYSIS_RADII[s]]
fig, axes = plt.subplots(1, len(pairs), figsize=(2.6 * len(pairs), 3.2),
                         squeeze=False)
for ax, (shot, x0) in zip(axes[0], pairs):
    for axis in ("Te", "ne"):
        ax.plot(scales, curves[(shot, x0, axis)], "-", color=COL[axis],
                lw=1.3, label=axis)
    ax.set_title(f"{shot}  x0={x0}", fontsize=9)
    ax.set_xlabel("scale_height"); ax.tick_params(labelsize=7)
axes[0][0].set_ylabel("a/L_pe")
axes[0][-1].legend(fontsize=6)
fig.suptitle("KBM-relevant drive reachable by each height axis — flat means unreachable")
plt.tight_layout(); plt.show()

## Absolute-unit bounds — Boyle 2011, converted per discharge

The handover: take the survey bounds in the units they are quoted in, convert
them to the scale factors this discharge's fit needs, and check the target is
reachable before a campaign spends CHEASE time discovering it is not. Scale
factors are an implementation detail of the transform — they differ per
discharge and per variable, and quoting a scan box in them is how the last
three rounds of confusion started.

Widths are Boyle 2011 PPCF (lithium scan, Fig. 7), which separates two regimes
— different plasmas, not one range, so they are scanned as separate boxes.

The metric per axis is the physical quantity the bound is quoted in: pedestal
-top value for heights, full width in %$\psi_N$ for widths, pedestal-top ratio
for `Ti_Te`. The scale→metric map is linear to machine precision, so one probe
point inverts it exactly: `scale = 1 + (target/nominal - 1)/slope`.

In [ ]:
# Boyle 2011 PPCF, Fig. 7: pedestal FULL widths in % psi_N.
#   7a = Delta_ne, 7d = Delta_Te, 7g = Delta_pe
BOYLE_WIDTHS = {
    "ELMy":     {"dne": (6, 12),  "dTe": (4, 7),   "dpe": (4, 8)},
    "ELM_free": {"dne": (14, 22), "dTe": (7, 10),  "dpe": (8, 12)},
}

# Pedestal-top values and the ion/electron ratio; frozen as one box because
# together with the widths they set beta.
TARGETS = {"Te_ped": (0.2, 0.8),   # keV at B0 ~ 0.4 T
           "ne_ped": (3.0, 7.0),   # 1e19 m^-3
           "Ti_Te":  (1.0, 2.0)}

# "auto" classifies each discharge from its own nominal widths: the regime is a
# property of the plasma. Scored against ELMy, 129038 -- ne 18.6%, pe 9.0%,
# squarely ELM-free -- produced width scale factors of 0.20-0.37, an
# instruction to shrink an ELM-free pedestal to a fifth of its width.
REGIME = "auto"          # "auto", "ELMy", "ELM_free", or {shot: regime}
# +/-30% on the scaled profile (user, 2026-08-23). Two independent reasons that
# agree: the survey spec's concrete reference is Hatch's +/-10-30% grid around
# the experimental pre-ELM state, and nothing beyond ~1.4 has ever been through
# cheaseBS -- the reshape-limit test that would establish more is still the open
# Phase-0 item. A target needing more than this is reported out of reach rather
# than quietly emitted.
SCALE_SANITY = (0.7, 1.3)

# Core drift. These axes are meant to move the PEDESTAL, but Stefanikova's core
# Gaussian is anchored to a_height at r=0 only, so scaling b_height or b_width
# drags the core too -- and for Te it drags it HARDER than the pedestal top
# (129015: scale_height 1.1 moves the pedestal top 8% and the core 14%). That is
# a property of the transform on these fits, not of one bad corner, so it is
# REPORTED by default rather than enforced: enforcing it at any sane threshold
# empties every Te box and that is a campaign decision, not a notebook default.
# Set CORE_ENFORCE = True to make it a hard filter on the box edges.
CORE_RHO   = 0.5            # "core" for this measurement
CORE_FLOOR = 0.05           # always tolerate this much fractional drift
CORE_FRAC  = 0.30           # ... or this fraction of the pedestal-top change
CORE_ENFORCE = False        # report only; True = trim box edges on it too

# axis -> (variable, transform kwarg, metric, unit, display factor)
#
# Te and ne only, height and width each. That is what this campaign needs: the
# KBM drive is the electron pressure gradient, which these four axes set between
# them, and Boyle's scan is quoted in exactly these quantities (Delta_ne,
# Delta_Te, Delta_pe and the pedestal-top values).
#
# Ti is deliberately NOT an axis, and is not needed for either purpose:
#   - scaling ne already rewrites ni and nz through quasineutrality inside
#     DischargePhysics, so ion density follows electron density;
#   - Ti/Te = 1-2 is a real axis of the FULL survey spec, but it is a separate
#     physics knob (ion drive, beta partition) rather than part of matching
#     Boyle, and its metric is the least trustworthy of the five: 132588 reads
#     3.2 against a 1.0-2.0 target because that discharge's Ti and Te pedestals
#     sit far apart, and the axis barely moves it (slope 0.06).
# Set SCAN_TI_TE = True for a Ti-focused campaign.
SCAN_TI_TE = False

AXES = {
    "Te_ped_scale":   ("Te", "scale_height", "ped_top", "keV",   1e-3),
    "ne_ped_scale":   ("ne", "scale_height", "ped_top", "1e19",  1e-19),
    "Te_width_scale": ("Te", "scale_width",  "width",   "%psiN", 1.0),
    "ne_width_scale": ("ne", "scale_width",  "width",   "%psiN", 1.0),
}
if SCAN_TI_TE:
    AXES["Ti_Te_scale"] = ("Ti", "scale_height", "ti_te", "-", 1.0)


def _apply(phys, var, fit, kwarg, s):
    return phys.apply_mtanh_full(var, fit=fit, **{kwarg: s},
                                 enforce_quasineutrality=True, qz=6.0)


PED_TOP_R = {}


def ped_top_radius(shot, var):
    """Radius of this variable's fitted pedestal top, b_pos - 2*b_width.

    Per VARIABLE, not per discharge: ne and Te pedestals sit 0.02-0.05 apart.
    Held fixed at the nominal fit's value and reused for every scaled profile,
    so the metric measures the profile moving rather than the measuring point
    moving with it.

    The earlier version read the value at the discharge's measured |grad pe|
    region edge, which is a pedestal-top radius only when the pedestal is
    narrow. On 132588 that edge is 0.560 and the value there is 0.96 keV -- a
    core value. scale_height cannot move it, so the axis reported slope -0.003
    and was declared unreachable when it is in fact perfectly usable.
    """
    key = (shot, var)
    if key not in PED_TOP_R:
        fp = fit_mtanh_full(results[shot]["phys"].ds, var,
                            **FIT_KWARGS)[1]["fit_params"]
        PED_TOP_R[key] = float(np.clip(fp["b_pos"] - 2.0 * fp["b_width"],
                                       GRAD_SEARCH[0], 0.98))
    return PED_TOP_R[key]


def core_drift(phys0, q, var, shot):
    """(core drift, budget) for a transformed profile, both fractional.

    Drift is the largest fractional change inside CORE_RHO; the budget is
    CORE_FRAC of the fractional change the axis produced at the pedestal top,
    floored at CORE_FLOOR. Drift above budget means the knob moved the core
    more than it moved the thing it is named after.
    """
    x = np.asarray(phys0.rhot.values)
    y0 = np.asarray(phys0.ds[var].values, dtype=float)
    y = np.asarray(q.ds[var].values, dtype=float)
    r = ped_top_radius(shot, var) if var in ("Te", "ne") else 0.9
    i = int(np.argmin(np.abs(x - r)))
    ped_change = abs(y[i] / y0[i] - 1.0)
    m = x <= CORE_RHO
    drift = float(np.max(np.abs(y[m] / y0[m] - 1.0)))
    return drift, max(CORE_FLOOR, CORE_FRAC * ped_change)


def _ped_top(phys, var, shot, at=None):
    """Profile value at a fitted pedestal-top radius."""
    x = np.asarray(phys.rhot.values)
    y = np.asarray(phys.ds[var].values, dtype=float)
    r = ped_top_radius(shot, at or var)
    return float(y[int(np.argmin(np.abs(x - r)))])


def _width_psin(phys, var):
    """Full pedestal width in % psi_N.

    Boyle quotes widths in poloidal flux, the fit works in rho_tor, and the
    conversion factor is not a constant — it depends where the pedestal sits.
    Stefanikova's b_width is a QUARTER width (full = 4*b_width) about b_pos.
    """
    ds = phys.ds.copy()
    if var == "pe":
        ds["pe"] = phys.ds["ne"] * phys.ds["Te"]
    fp = fit_mtanh_full(ds, var, **FIT_KWARGS)[1]["fit_params"]
    rhot = np.asarray(phys.rhot.values)
    rhop = np.asarray(phys.rhop.values)
    w = 4.0 * fp["b_width"]
    pl, ph = np.interp([fp["b_pos"] - w / 2, fp["b_pos"] + w / 2], rhot, rhop)
    return 100.0 * (ph ** 2 - pl ** 2)


def metric_of(phys, var, kind, shot):
    if kind == "ped_top":
        return _ped_top(phys, var, shot)
    if kind == "width":
        return _width_psin(phys, var)
    if kind == "ti_te":
        # both at the Te pedestal top, so the ratio is taken at one radius
        return (_ped_top(phys, "Ti", shot, at="Te")
                / _ped_top(phys, "Te", shot, at="Te"))
    raise ValueError(kind)


def axis_response(phys, axis, fits, shot, probe=1.2):
    """Nominal metric (in the units the bounds are quoted in) and its linear
    slope in the scale factor. The slope is relative, so unit conversion does
    not touch it; the nominal must be converted or the inversion is meaningless."""
    var, kwarg, kind, unit, disp = AXES[axis]
    at = lambda s: metric_of(_apply(phys, var, fits[var], kwarg, s), var, kind, shot)
    m0 = at(1.0)
    return m0 * disp, (at(probe) - m0) / m0 / (probe - 1.0), unit, disp


def scale_for(m0, slope, target):
    """Invert the linear map; nan when the axis cannot move the metric."""
    if not np.isfinite(slope) or abs(slope) < 1e-6:
        return float("nan")
    return 1.0 + ((target / m0) - 1.0) / slope


def classify_regime(phys):
    """Boyle band the nominal widths sit in. Votes across dTe, dne, dpe."""
    widths = {"dTe": _width_psin(phys, "Te"), "dne": _width_psin(phys, "ne"),
              "dpe": _width_psin(phys, "pe")}
    scores = {}
    for name, bands in BOYLE_WIDTHS.items():
        inside = sum(bands[k][0] <= w <= bands[k][1] for k, w in widths.items())
        dist = sum(0.0 if bands[k][0] <= w <= bands[k][1]
                   else min(abs(w - bands[k][0]), abs(w - bands[k][1]))
                   for k, w in widths.items())
        scores[name] = (inside, -dist)
    return max(scores, key=scores.get), widths, scores


print("axes:", ", ".join(AXES))

### Nominal values against the target box

`nominal` is where each discharge already sits; `need lo` / `need hi` are the
scale factors that would reach the bound. A nominal inside its target range is
the comfortable case; outside means the axis has to travel one way only.

The chart under the table is the one to read. Each panel is **the measured
metric against scale factor**, with the target band shaded, the nominal starred,
and dashed lines where the target needs the axis to go. Two things are visible
there that no column of the table shows:

- **whether the response is linear.** The inversion `scale = 1 + (target/nominal
  - 1)/slope` assumes it is, and takes one probe point on faith. Dots sitting on
  the grey line mean the assumption holds; dots leaving it mean the emitted
  scale factor is the solution to a map that is not there. 132588 `ne_width` is
  the case to look at.
- **how far the band is, in axis units.** A slope of 0.9 versus 0.5 reads the
  same in a table; here you see the panel where the band is unreachable because
  the line is too shallow to climb, versus the panel where it is unreachable
  because the band sits nowhere near the nominal (132588 `Ti_Te`, flat at 3.2
  against a 1.0–2.0 target).

In [ ]:
SHOT_REGIME, FITS, WIDTHS = {}, {}, {}
print("regime classification:")
for shot in shots:
    phys = results[shot]["phys"]
    FITS[shot] = {v: fit_mtanh_full(phys.ds, v, **FIT_KWARGS)[0]
                  for v in {AXES[a][0] for a in AXES}}
    if isinstance(REGIME, dict):
        SHOT_REGIME[shot] = REGIME[shot]
        continue
    if REGIME != "auto":
        SHOT_REGIME[shot] = REGIME
        continue
    best, widths, scores = classify_regime(phys)
    SHOT_REGIME[shot], WIDTHS[shot] = best, widths
    tie = len({v[0] for v in scores.values()}) == 1
    pinned = [v for v in ("Te", "ne")
              if results[shot]["vars"][v].get("pinned")]
    print(f"  {shot}: {best:<9} ("
          + ", ".join(f"{k} {v:.1f}%" for k, v in widths.items()) + ";  "
          + ", ".join(f"{n} {s[0]}/3 in" for n, s in scores.items()) + ")"
          + ("   TIE — broken by distance, check by hand" if tie else "")
          + (f"   width unreliable: {'/'.join(pinned)} b_pos on bound"
             if pinned else ""))

bounds_per_shot, reach_rows = {}, []
for shot in shots:
    phys = results[shot]["phys"]
    per_axis = {}
    for axis in AXES:
        var, kwarg, kind, unit, disp = AXES[axis]
        if kind == "width":
            lo, hi = BOYLE_WIDTHS[SHOT_REGIME[shot]]["dTe" if var == "Te" else "dne"]
        else:
            lo, hi = TARGETS["Ti_Te" if kind == "ti_te" else f"{var}_ped"]
        m0, slope, unit, disp = axis_response(phys, axis, FITS[shot], shot)
        s_lo, s_hi = sorted((scale_for(m0, slope, lo), scale_for(m0, slope, hi)))
        # What the axis actually reaches inside the +/-30% cap, MEASURED at the
        # two edges rather than extrapolated. With the cap this tight the useful
        # question stopped being "can it reach the band" -- almost nothing can
        # span a Boyle band in 30% -- and became "how much of the band does it
        # cover, and from which side".
        edges = tuple(sorted(
            metric_of(_apply(phys, var, FITS[shot][var], kwarg, sc), var, kind,
                      shot) * disp for sc in SCALE_SANITY))
        overlap = max(0.0, min(edges[1], hi) - max(edges[0], lo))
        per_axis[axis] = {"nominal": m0, "unit": unit, "slope": slope,
                          "target": (lo, hi), "scale": (s_lo, s_hi),
                          "reach": edges,
                          "cover": overlap / (hi - lo) if hi > lo else np.nan,
                          "ok": overlap > 0}
        reach_rows.append({"shot": shot, "axis": axis, **per_axis[axis]})
    bounds_per_shot[shot] = per_axis

hdr = (f"{'shot':>7} {'axis':<16} {'nominal':>9} {'target':>11} "
       f"{f'reach at {SCALE_SANITY}':>19} {'unit':<6} {'covers':>7}  in band?")
print(); print(hdr); print("-" * len(hdr))
for r in reach_rows:
    lo, hi = r["target"]
    # An axis whose own fit has b_pos on the bound reports a metric taken off a
    # pedestal the fitter never located -- 132588 ne is the case, and it is why
    # that axis shows a 35.7 %psiN width.
    pin = ("  [fit pinned]"
           if results[r["shot"]]["vars"][AXES[r["axis"]][0]].get("pinned") else "")
    inside = r["target"][0] <= r["nominal"] <= r["target"][1]
    print(f"{r['shot']:>7} {r['axis']:<16} {r['nominal']:>9.3f} "
          f"{f'{lo}-{hi}':>11} {r['reach'][0]:>8.3f}-{r['reach'][1]:<10.3f} "
          f"{r['unit']:<6} {r['cover']:>6.0%}  "
          f"{'nominal in band' if inside else 'nominal OUTSIDE band'}{pin}")
print(f"\ncovers = fraction of the Boyle/target band this axis can reach within "
      f"{SCALE_SANITY}, measured at the edges. A band is a population of "
      "discharges, not a target for one discharge to span: 100% would mean one "
      "plasma covering Boyle's whole scan, which is not the goal. What matters "
      "is that the reachable interval overlaps the band and moves in the right "
      "direction.")
print("height metrics are read at the fitted pedestal top, b_pos - 2*b_width, "
      "held fixed at the nominal fit:")
for shot in shots:
    print(f"  {shot}  " + "  ".join(f"{v} {ped_top_radius(shot, v):.3f}"
                                    for v in ("Te", "ne")))
print("Ti_Te is taken at the Te radius for both species. Where the Ti and Te "
      "pedestals sit far apart (132588) the ratio there is a real feature of "
      "the discharge, not a fit artefact -- but the axis moves it only weakly.")

# ---------------------------------------------------------------------------
# The response curves the table above summarises in three numbers each.
# Measured points against the linear model that the inversion assumes: if the
# dots leave the line, scale_for() is inverting a map that is not there.
# ---------------------------------------------------------------------------
PROBE_SCALES = np.array([0.7, 0.85, 1.0, 1.15, 1.3])   # the allowed range

fig, axr = plt.subplots(len(AXES), len(shots),
                        figsize=(3.2 * len(shots), 2.1 * len(AXES)),
                        squeeze=False)
for i, axis in enumerate(AXES):
    var, kwarg, kind, unit, disp = AXES[axis]
    for j, shot in enumerate(shots):
        ax = axr[i][j]
        r = bounds_per_shot[shot][axis]
        lo, hi = r["target"]
        phys = results[shot]["phys"]
        meas = [metric_of(_apply(phys, var, FITS[shot][var], kwarg, s), var,
                          kind, shot) * disp for s in PROBE_SCALES]
        model = r["nominal"] * (1 + r["slope"] * (PROBE_SCALES - 1))
        ax.axhspan(lo, hi, color="tab:green", alpha=0.15, label="Boyle/target")
        ax.plot(PROBE_SCALES, model, "-", color="0.6", lw=1.0, label="linear model")
        ax.plot(PROBE_SCALES, meas, "o", ms=4, color=COL.get(var, "k"),
                label="measured")
        ax.plot([1.0], [r["nominal"]], "*", ms=10, color="k", label="nominal")
        # the scale factors the target asks for, before any physics trimming;
        # off-plot when the axis cannot get there at all
        for s_edge in r["scale"]:
            if np.isfinite(s_edge) and PROBE_SCALES[0] <= s_edge <= PROBE_SCALES[-1]:
                ax.axvline(s_edge, color="tab:purple", lw=1.0, ls="--")
        ax.tick_params(labelsize=7)
        if i == 0:
            ax.set_title(str(shot), fontsize=10)
        if j == 0:
            ax.set_ylabel(f"{axis}\n[{unit}]", fontsize=7)
        if i == len(AXES) - 1:
            ax.set_xlabel("scale factor")
axr[0][-1].legend(fontsize=5)
fig.suptitle("axis response: measured metric vs scale, target band shaded, "
             "dashed = scale factors the target needs")
plt.tight_layout(); plt.show()

### The scan box, in scale factors, ready for ScanStudy

The box is emitted per discharge and drawn underneath: **grey is what the
target asks for, colour is what survives the physics filters**, so a short
colour bar against a long grey one is an axis the filters cut down and a missing
one is an axis dropped entirely.

Edges are trimmed by physics rather than clipped at a round number: an edge is
walked back toward nominal until the profile it produces is one Boyle could
have observed — positive, monotonic outward through the pedestal, with a $p_e$
width in band, **and with the core left alone**.

That last test is new and it is the one that bites. These axes are meant to
move the pedestal, but Stefanikova's core Gaussian is anchored to `a_height` at
$r=0$ only, so scaling `b_height` or `b_width` drags the core with it: 129015
`Te` at scale 3.0 lifts the on-axis value from 0.8 to 1.8 keV and grows a hump
at $\rho_t\approx0.15$, and its `ne_width` at 2.45 raises core density by 65%.
Both passed every earlier check, because those only looked at 0.6–1.0. A corner
is now rejected when the core moves by more than 30% of the pedestal-top change
it was asked for. `pe` gets no axis (it is derived, and CHEASE builds the
pressure from the profiles regardless) but Boyle's panel 7g makes it a free
consistency filter on the corners.

In [ ]:
def edge_ok(phys0, fits, axis, s, shot):
    """Is this box edge a plasma worth submitting?

    Physical tests, not an arbitrary scale ceiling: a scale factor is out of
    bounds when the profile it produces stops being one Boyle observed.
    """
    var, kwarg, kind, _, _ = AXES[axis]
    try:
        q = _apply(phys0, var, fits[var], kwarg, s)
        y = np.asarray(q.ds[var].values, dtype=float)
        if float(np.min(y)) <= 0:
            return False                       # went non-positive
        # Monotonic decrease outward through the pedestal. A large height
        # scaling can push the Stefanikova core Gaussian and the pedestal
        # plateau out of proportion and raise a bump at rho~0.9 that the width
        # filter cannot see -- a bump changes shape without changing width.
        x = np.asarray(q.rhot.values)
        m = (x >= 0.6) & (x <= 1.0)
        if np.any(np.diff(y[m]) > 0.02 * float(np.max(y[m]))):
            return False
        if CORE_ENFORCE:
            c, budget = core_drift(phys0, q, var, shot)
            if c > budget:
                return False
        w = _width_psin(q, "pe")
    except Exception:
        return False
    lo_pe, hi_pe = BOYLE_WIDTHS[SHOT_REGIME[shot]]["dpe"]
    if lo_pe <= w <= hi_pe:
        return True
    # Some nominals start outside Boyle's band. Demanding the band outright
    # would empty their boxes and say nothing; the test becomes "do not make it
    # worse", leaving out-of-band-at-nominal as the separate finding it is.
    nom = WIDTHS.get(shot, {}).get("dpe")
    if nom is None or lo_pe <= nom <= hi_pe:
        return False
    dev = min(abs(w - lo_pe), abs(w - hi_pe))
    return dev <= min(abs(nom - lo_pe), abs(nom - hi_pe)) * 1.25


def trim_edge(phys0, fits, axis, s_target, shot, s_from=1.0, tol=0.01):
    """Bisect an edge back toward nominal until it passes edge_ok."""
    if edge_ok(phys0, fits, axis, s_target, shot):
        return s_target
    lo, hi = s_from, s_target
    if not edge_ok(phys0, fits, axis, lo, shot):
        return None
    while abs(hi - lo) > tol:
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if edge_ok(phys0, fits, axis, mid, shot) else (lo, mid)
    return round(lo, 4)


SG_BOUNDS = {}
for shot in shots:
    phys0, fits = results[shot]["phys"], FITS[shot]
    WIDTHS.setdefault(shot, {}).setdefault("dpe", _width_psin(phys0, "pe"))
    box, notes = {}, []
    for axis, r in bounds_per_shot[shot].items():
        sl, sh = r["scale"]
        if not (np.isfinite(sl) and np.isfinite(sh)):
            notes.append(f"{axis}: axis cannot move its metric — dropped")
            continue
        # Clip to the tested scale range first: nothing outside it has been run
        # against cheaseBS, so trimming from an untested edge would bisect
        # through profiles we have no basis to judge.
        sl, sh = max(sl, SCALE_SANITY[0]), min(sh, SCALE_SANITY[1])
        if sh <= sl:
            notes.append(f"{axis}: target lies outside the tested scale range "
                         f"{SCALE_SANITY} — dropped")
            continue
        cl = trim_edge(phys0, fits, axis, sl, shot)
        ch = trim_edge(phys0, fits, axis, sh, shot)
        if cl is None or ch is None or ch <= cl:
            notes.append(f"{axis}: no usable range — even small moves off "
                         "nominal leave Boyle's pe band")
            continue
        if (cl, ch) != (sl, sh):
            lo, hi = r["target"]
            notes.append(
                f"{axis}: trimmed to physics, reaches "
                f"{r['nominal'] * (1 + r['slope'] * (cl - 1)):.3g}-"
                f"{r['nominal'] * (1 + r['slope'] * (ch - 1)):.3g} {r['unit']} "
                f"of {lo}-{hi}")
        box[axis] = (round(cl, 4), round(ch, 4))
    SG_BOUNDS[shot] = box
    print(f"{shot}: {json.dumps(box)}")
    for n in notes:
        print(f"    ! {n}")

with open("sg_bounds.json", "w") as fh:
    json.dump({"form": "full",
               "regime": {str(k): v for k, v in SHOT_REGIME.items()},
               "targets": TARGETS, "boyle_widths": BOYLE_WIDTHS,
               "scale_sanity": list(SCALE_SANITY),
               "bounds": {str(k): v for k, v in SG_BOUNDS.items()}}, fh, indent=1)
print("\nwritten: sg_bounds.json")

# ---------------------------------------------------------------------------
# The box, drawn. Grey = what the target asks for before trimming, colour =
# what is emitted. A missing colour bar is a dropped axis; a colour bar much
# shorter than its grey one is an axis the physics filters cut down.
# ---------------------------------------------------------------------------
fig, axb = plt.subplots(1, len(shots), figsize=(3.4 * len(shots), 3.2),
                        sharex=True, squeeze=False)
axis_names = list(AXES)
for j, shot in enumerate(shots):
    ax = axb[0][j]
    for k, axis in enumerate(axis_names):
        y = len(axis_names) - 1 - k
        rq = bounds_per_shot[shot][axis]["scale"]
        if all(np.isfinite(rq)):
            ax.plot(rq, [y + 0.16] * 2, "-", lw=5, color="0.82",
                    solid_capstyle="butt")
        box = SG_BOUNDS[shot].get(axis)
        if box:
            var = AXES[axis][0]
            ax.plot(box, [y - 0.08] * 2, "-", lw=6, color=COL.get(var, "0.3"),
                    solid_capstyle="butt")
            ax.plot(box, [y - 0.08] * 2, "|", ms=9, color="k")
        else:
            ax.text(1.0, y - 0.08, "dropped", fontsize=6, ha="center",
                    va="center", color="tab:red")
    ax.axvline(1.0, color="k", lw=0.8)
    ax.axvspan(*SCALE_SANITY, color="tab:blue", alpha=0.05)
    ax.set_yticks(range(len(axis_names)))
    ax.set_yticklabels([a.replace("_scale", "") for a in axis_names[::-1]],
                       fontsize=7)
    ax.set_xlim(SCALE_SANITY[0] - 0.15, SCALE_SANITY[1] + 0.15)
    ax.set_xlabel("scale factor")
    ax.set_title(f"{shot}  {SHOT_REGIME[shot]}", fontsize=9)
    ax.tick_params(labelsize=7)
fig.suptitle("scan box per discharge — grey: asked for by the target, "
             "colour: emitted after physics trimming")
plt.tight_layout(); plt.show()

### Corners: pressure-width check and a look before handing over

A corner whose scanned `ne` and `Te` widths imply a $p_e$ width outside Boyle's
band is not a plasma he observed — a box edge to pull in, not a run to submit.

Three figures follow: the two filters drawn (pe width against each discharge's
Boyle band, and core drift against its budget), then the corner gallery.

The gallery draws every corner over the **full radius**, in `Te`, `ne` and
the derived `pe`, because that is the object CHEASE-BS is handed. Scaling `ne`
rewrites `ni` and `nz` through quasineutrality inside `DischargePhysics`, so the
`pe` row is read off the transformed object rather than reconstructed by hand.

Read it for three things: curves that **separate visibly** (a corner sitting on
nominal is an axis contributing nothing), curves that **stay physical** across
the whole radius (no crossing, no core bump, nothing going negative), and a
spread that **matches what the bounds table claims**.

The `core` / `budget` columns quantify what the gallery shows. `core` is the
largest fractional change inside $\rho_t<0.5$; `budget` is 30% of the
fractional change the axis produced at its own pedestal top. Over budget means
the knob moved the core more than the thing it is named after — 129015
`Te_ped_scale` at 3.0 lifts the on-axis value from 0.8 to 1.8 keV and grows a
hump at $\rho_t\approx0.15$.

**Reported, not enforced.** This is a property of scaling a Stefanikova fit,
not of one bad corner: on 129015 `Te`, `scale_height` 1.1 already moves the
pedestal top 8% and the core 14%. Enforcing it at any defensible threshold
empties every `Te` box, so the decision belongs to the campaign — flip
`CORE_ENFORCE` to make it a hard filter on the box edges.

In [ ]:
print(f"{'shot':>7} {'corner':<26} {'pe width':>9}  band   {'core':>5} "
      f"{'budget':>6}  verdict")
print("-" * 80)
pe_rows = []
for shot in shots:
    phys0, fits = results[shot]["phys"], FITS[shot]
    lo_pe, hi_pe = BOYLE_WIDTHS[SHOT_REGIME[shot]]["dpe"]
    corners = [("nominal", {})]
    for axis, (blo, bhi) in SG_BOUNDS[shot].items():
        if AXES[axis][0] in ("Te", "ne"):
            corners += [(f"{axis} lo ({blo:.2f})", {axis: blo}),
                        (f"{axis} hi ({bhi:.2f})", {axis: bhi})]
    for label, point in corners:
        phys = phys0
        for axis, s in point.items():
            var, kwarg, _, _, _ = AXES[axis]
            phys = _apply(phys, var, fits[var], kwarg, s)
        try:
            w = _width_psin(phys, "pe")
        except Exception as exc:
            print(f"{shot:>7} {label:<26} {'--':>9}  fit failed: {type(exc).__name__}")
            continue
        ok = lo_pe <= w <= hi_pe
        if point:
            drift, budget = core_drift(phys0, phys, AXES[next(iter(point))][0],
                                       shot)
        else:
            drift, budget = 0.0, float("nan")
        pe_rows.append({"shot": shot, "corner": label, "pe_width": w, "ok": ok,
                        "core_drift": drift, "core_budget": budget})
        bud = f"{budget*100:>5.0f}%" if budget == budget else f"{'--':>6}"
        print(f"{shot:>7} {label:<26} {w:>8.1f}%  {lo_pe:>2}-{hi_pe:<2} "
              f"{drift*100:>5.0f}% {bud}  "
              f"{'ok' if ok else 'OUTSIDE Boyle'}"
              + ("   CORE MOVES" if drift > budget else ""))
bad = sum(not r["ok"] for r in pe_rows)
over = sum(r["core_drift"] > r["core_budget"] for r in pe_rows)
print(f"\n{bad}/{len(pe_rows)} corners outside their discharge's pe width band; "
      f"{over}/{len(pe_rows)} move the core by more than {CORE_FRAC:.0%} of the "
      "pedestal-top change they were asked for.")
print("Core drift is REPORTED, not enforced (CORE_ENFORCE = False): it is a "
      "property of scaling a Stefanikova fit, not of one bad corner. On 129015 "
      "Te, scale_height 1.1 already moves the pedestal top 8% and the core 14%, "
      "so enforcing it empties every Te box -- that call belongs to the "
      "campaign, not to this notebook's defaults.")

# ---------------------------------------------------------------------------
# The same two tests, drawn. Left: pe width per corner against the discharge's
# Boyle band. Right: core drift against its budget -- anything above the
# diagonal moved the core more than the pedestal it was asked to move.
# ---------------------------------------------------------------------------
fig, (axw, axc) = plt.subplots(1, 2, figsize=(12, 0.32 * len(pe_rows) + 2.2),
                               gridspec_kw={"width_ratios": [1.35, 1]})
ypos = np.arange(len(pe_rows))[::-1]
for y, r in zip(ypos, pe_rows):
    lo_pe, hi_pe = BOYLE_WIDTHS[SHOT_REGIME[r["shot"]]]["dpe"]
    axw.plot([lo_pe, hi_pe], [y, y], "-", lw=7, color="tab:green", alpha=0.16,
             solid_capstyle="butt")
    axw.plot(r["pe_width"], y, "o" if r["corner"] == "nominal" else "s", ms=6,
             color="k" if r["corner"] == "nominal" else
             ("tab:blue" if r["ok"] else "tab:red"))
axw.set_yticks(ypos)
axw.set_yticklabels([f"{r['shot']} {r['corner']}" for r in pe_rows], fontsize=6)
axw.set_xlabel("pe pedestal width  [%psiN]")
axw.set_title("corner pe width vs this discharge's Boyle band "
              "(circle = nominal)", fontsize=9)
axw.grid(axis="x", alpha=0.3)

mk = {s: m for s, m in zip(shots, ("o", "s", "^", "D"))}
for r in pe_rows:
    if r["corner"] == "nominal":
        continue
    axc.plot(100 * r["core_budget"], 100 * r["core_drift"], mk[r["shot"]], ms=6,
             color="tab:red" if r["core_drift"] > r["core_budget"] else "tab:green",
             label=str(r["shot"]))
lim = [0, 1.05 * 100 * max(max(r["core_drift"] for r in pe_rows),
                           max(r["core_budget"] for r in pe_rows
                               if r["core_budget"] == r["core_budget"]))]
axc.plot(lim, lim, "k--", lw=1.0)
axc.set_xlabel("core-drift budget  [%]"); axc.set_ylabel("core drift  [%]")
axc.set_title("above the diagonal = the knob moved the core more\n"
              "than the pedestal it is named after", fontsize=9)
axc.set_xlim(*lim); axc.set_ylim(*lim); axc.grid(alpha=0.3)
# marker = discharge (black in the legend, since the fill colour carries
# pass/fail and would otherwise be read as a discharge)
axc.legend(handles=[plt.Line2D([], [], ls="", marker=mk[s], color="k",
                               label=str(s)) for s in shots], fontsize=6)
plt.tight_layout(); plt.show()

GAL_VARS = ("Te", "ne", "pe")
fig, axg = plt.subplots(len(GAL_VARS), len(shots),
                        figsize=(3.6 * len(shots), 2.6 * len(GAL_VARS)),
                        squeeze=False)
for j, shot in enumerate(shots):
    phys, fits = results[shot]["phys"], FITS[shot]
    x = np.asarray(phys.rhot.values)

    def prof(q, var):
        """Te/ne straight off the dataset; pe derived, since that is how CHEASE
        builds the pressure and how Boyle's panel 7g is quoted."""
        if var == "pe":
            return (np.asarray(q.ds["ne"].values, dtype=float)
                    * np.asarray(q.ds["Te"].values, dtype=float))
        return np.asarray(q.ds[var].values, dtype=float)

    for row, var in enumerate(GAL_VARS):
        ax = axg[row][j]
        ax.plot(x, prof(phys, var), "-", lw=2.2, color="k", label="nominal",
                zorder=3)
        for axis, (lo, hi) in SG_BOUNDS[shot].items():
            v, kwarg = AXES[axis][0], AXES[axis][1]
            if v not in ("Te", "ne", "Ti"):
                continue
            for sc, ls in ((lo, "--"), (hi, ":")):
                q = _apply(phys, v, fits[v], kwarg, sc)
                # scaling ne rewrites ni/nz through quasineutrality, so pe and
                # the other rows have to be read off the transformed object
                ax.plot(x, prof(q, var), ls, lw=1.1,
                        color=COL.get(v, "0.4"), alpha=0.9,
                        label=f"{axis.replace('_scale','')} {sc:.2f}")
        ax.axvspan(*results[shot]["win"], color="tab:green", alpha=0.08)
        for r0 in ANALYSIS_RADII[shot]:
            ax.axvline(r0, color="k", lw=0.8, ls=":", alpha=0.7)
        ax.set_xlim(0.0, 1.0)
        ax.set_ylabel(var); ax.tick_params(labelsize=7)
        if row == 0:
            ax.set_title(f"{shot}  {SHOT_REGIME[shot]}", fontsize=10)
            ax.legend(fontsize=5, ncol=2)
        if row == len(GAL_VARS) - 1:
            ax.set_xlabel("rho_tor")
fig.suptitle("scan box corners, full radius — every curve is a profile CHEASE-BS "
             "would be handed")
plt.tight_layout(); plt.show()